# S'mile — travel price models by time of day

Models that predict a **price range** (p10 / p50 / p90) for transport whose price changes during the day:

| Model | Training data (public) | Target |
|---|---|---|
| Rideshare | NYC TLC High-Volume For-Hire trip records (Uber/Lyft), two months sampled | base passenger fare (before tax, tolls, fees, tip) |
| Airfare (route level) | U.S. DOT Consumer Airfare Report, Table 1a (all U.S. airport-pair markets) | average fare for the airport pair and quarter |
| Flight time of day | Expedia domestic fares, New Zealand, Sep–Dec 2019 (Mendeley Data, CC BY 4.0) | fare relative to the same route's median that day |

The model produces evidence and plain code decides: the app shows predicted ranges as estimates, never as the exact amount a traveler approves.

Validation is temporal (train on the past, test on the most recent period) against a simple baseline, and every model is exported to compact JSON and re-scored in pure Python to prove the TypeScript evaluator has the same numbers to match.

In [1]:
import os, sys, json, time, math, urllib.request, warnings
# This server has system copies of numexpr/bottleneck built for NumPy 1.x; pandas only uses them
# as optional speedups, so mark them unavailable instead of letting them print import errors.
sys.modules["numexpr"] = None; sys.modules["bottleneck"] = None
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

ROOT = os.path.expanduser("~/smile-price-models"); DATA = f"{ROOT}/data"; OUT = f"{ROOT}/models"
os.makedirs(DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)
SEED = 7
DEVICE = "cuda" if xgb.build_info().get("USE_CUDA") else "cpu"
QUANTILES = (0.1, 0.5, 0.9)
print("xgboost", xgb.__version__, "| device", DEVICE, "| pandas", pd.__version__, "| numpy", np.__version__)

xgboost 3.2.0 | device cuda | pandas 2.3.3 | numpy 2.4.2


In [2]:
def download(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    tmp = path + ".part"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (smile-price-models notebook)"})
    with urllib.request.urlopen(req, timeout=300) as r, open(tmp, "wb") as f:
        while True:
            b = r.read(1 << 22)
            if not b:
                break
            f.write(b)
    os.replace(tmp, path)
    return path

def train_quantiles(Xtr, ytr, Xva, yva, params):
    models = {}
    for q in QUANTILES:
        m = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=q, tree_method="hist", device=DEVICE, random_state=SEED, **params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        models[q] = m
    return models

def compact(model):
    """XGBoost trees -> compact JSON the app evaluates in TypeScript (no Python at runtime)."""
    j = json.loads(model.get_booster().save_raw("json").decode())
    learner = j["learner"]
    base = float(str(learner["learner_model_param"]["base_score"]).strip("[]").split(",")[0])
    trees = []
    for t in learner["gradient_booster"]["model"]["trees"]:
        trees.append({"l": t["left_children"], "r": t["right_children"], "f": t["split_indices"],
                      "c": [float(x) for x in t["split_conditions"]], "d": [int(x) for x in t["default_left"]]})
    return {"base": base, "trees": trees}

def predict_compact(cm, X):
    """Reference evaluator: go left when x < threshold (float32, as XGBoost does); missing follows default_left."""
    X = np.asarray(X, dtype=np.float32); out = np.full(len(X), cm["base"], dtype=np.float64)
    for t in cm["trees"]:
        L, R, F, C, D = t["l"], t["r"], t["f"], np.asarray(t["c"], dtype=np.float32), t["d"]
        for i, row in enumerate(X):
            n = 0
            while L[n] != -1:
                v = row[F[n]]
                n = (L[n] if D[n] else R[n]) if np.isnan(v) else (L[n] if v < C[n] else R[n])
            out[i] += float(C[n])
    return out

def interval_report(name, y, p10, p50, p90, baseline=None):
    r = {"n_test": int(len(y)), "mae_p50": float(mean_absolute_error(y, p50)), "r2_p50": float(r2_score(y, p50)),
         "mape_p50": float(np.mean(np.abs(p50 - y) / np.maximum(y, 1))),
         "coverage_p10_p90": float(np.mean((y >= p10) & (y <= p90))), "median_band_width": float(np.median(p90 - p10))}
    if baseline is not None:
        r["mae_baseline"] = float(mean_absolute_error(y, baseline)); r["r2_baseline"] = float(r2_score(y, baseline))
    print(name, json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in r.items()}))
    return r

## 1 · Rideshare fares (NYC TLC high-volume for-hire trips)

Real Uber and Lyft trips with the fare the rider was charged, including surge.

**Each period of the day has its own price.** Rush hours, late nights and weekends change both traffic and surge. Before a trip the app only knows the free-flow road distance (routing engines don't know tomorrow's traffic), so actual trip minutes are *not* a model input: the time features — hour, day of week, hour of week, named day period, weekend — learn what each part of the day does to the price. An ablation below measures how much they add.

In [3]:
def tlc_url(ym): return f"https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_{ym}.parquet"
def exists(url):
    try:
        with urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=30) as r:
            return r.status == 200
    except Exception:
        return False

cands = [f"{y}-{m:02d}" for y in (2026, 2025, 2024) for m in range(12, 0, -1)]
latest = next(ym for ym in cands if exists(tlc_url(ym)))
y, m = int(latest[:4]), int(latest[5:])
earlier = f"{y if m > 6 else y - 1}-{(m - 6) if m > 6 else (m + 6):02d}"
MONTHS = [earlier, latest]
print("training months:", MONTHS)
for ym in MONTHS:
    p = download(tlc_url(ym), f"{DATA}/fhvhv_{ym}.parquet"); print(ym, round(os.path.getsize(p) / 2**20), "MB")

training months: ['2026-02', '2026-08']
2026-02 455 MB
2026-08 479 MB


In [4]:
AIRPORT_ZONES = {1, 132, 138}   # EWR, JFK, LGA taxi zones
PERIODS = [("late night", 0, 5), ("early morning", 5, 7), ("morning rush", 7, 10), ("midday", 10, 16),
           ("evening rush", 16, 19), ("evening", 19, 22), ("night", 22, 24)]
def period_of(hour):
    return next(i for i, (_, a, b) in enumerate(PERIODS) if a <= hour < b)
PERIOD_OF_HOUR = np.array([period_of(h) for h in range(24)], dtype="float32")
PERIOD_NAMES = [n for n, _, _ in PERIODS]

cols = ["hvfhs_license_num", "request_datetime", "PULocationID", "DOLocationID", "trip_miles", "trip_time",
        "base_passenger_fare", "shared_request_flag"]
frames = []
for ym in MONTHS:
    df = pd.read_parquet(f"{DATA}/fhvhv_{ym}.parquet", columns=cols)
    df = df[df.hvfhs_license_num.isin(["HV0003", "HV0005"])]          # Uber, Lyft
    df = df.sample(n=min(len(df), 3_000_000), random_state=SEED)
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)
print("rows sampled:", len(raw))

hour = raw.request_datetime.dt.hour.values
dow = raw.request_datetime.dt.dayofweek.values
d = pd.DataFrame({
    "miles": raw.trip_miles.astype("float32"),
    "hour": hour.astype("float32"),
    "dow": dow.astype("float32"),
    "hour_of_week": (dow * 24 + hour).astype("float32"),
    "period": PERIOD_OF_HOUR[hour],
    "weekend": (dow >= 5).astype("float32"),
    "month": raw.request_datetime.dt.month.astype("float32"),
    "airport": (raw.PULocationID.isin(AIRPORT_ZONES) | raw.DOLocationID.isin(AIRPORT_ZONES)).astype("float32"),
    "shared": (raw.shared_request_flag == "Y").astype("float32"),
    "lyft": (raw.hvfhs_license_num == "HV0005").astype("float32"),
    "minutes": (raw.trip_time / 60).astype("float32"),     # for cleaning only, not a model input
    "fare": raw.base_passenger_fare.astype("float32"),
    "ts": raw.request_datetime,
})
ok = d.miles.between(0.3, 100) & d.minutes.between(2, 240) & d.fare.between(2.5, 500) & ((d.fare / d.miles) < 30)
d = d[ok].reset_index(drop=True)
d["period_name"] = d.period.astype(int).map(dict(enumerate(PERIOD_NAMES)))
print("rows after cleaning:", len(d), "| dropped", int((~ok).sum()))

print(d.assign(fpm=d.fare / d.miles).groupby(["period_name", "weekend"]).fpm.median().unstack().round(2)
      .rename(columns={0.0: "weekday $/mi", 1.0: "weekend $/mi"}).reindex(PERIOD_NAMES).to_string())

rows sampled: 6000000


rows after cleaning: 5938855 | dropped 61145


weekend        weekday $/mi  weekend $/mi
period_name                              
late night             5.29          5.88
early morning          5.56          4.96
morning rush           7.09          5.52
midday                 7.16          6.63
evening rush           7.46          6.94
evening                6.59          6.56
night                  5.95          6.30


In [5]:
TIME_FEATURES = ["hour", "dow", "hour_of_week", "period", "weekend"]
RS_FEATURES = ["miles"] + TIME_FEATURES + ["month", "airport", "shared", "lyft"]
NO_TIME = ["miles", "month", "airport", "shared", "lyft"]
cut = d.ts.max() - pd.Timedelta(days=7)          # temporal split: the most recent week is the test set
tr, te = d[d.ts < cut], d[d.ts >= cut]
va = tr.sample(frac=0.05, random_state=SEED); tr = tr.drop(va.index)
print("train", len(tr), "| valid", len(va), "| test", len(te), "| test starts", cut)

base = LinearRegression().fit(tr[["miles"]], tr.fare)   # a flat per-mile rate card: same price at every hour
print("rate card: $%.2f + $%.2f/mi" % (base.intercept_, base.coef_[0]))

params = dict(n_estimators=250, max_depth=7, learning_rate=0.08, subsample=0.8, max_bin=256)
t0 = time.time()
rs = train_quantiles(tr[RS_FEATURES], tr.fare, va[RS_FEATURES], va.fare, params)
print("trained in %.1fs on %s" % (time.time() - t0, DEVICE))
P = {q: rs[q].predict(te[RS_FEATURES]) for q in QUANTILES}
rs_metrics = interval_report("rideshare", te.fare.values, P[0.1], P[0.5], P[0.9], base.predict(te[["miles"]]))

nt = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=0.5, tree_method="hist", device=DEVICE, random_state=SEED, **params)
nt.fit(tr[NO_TIME], tr.fare, eval_set=[(va[NO_TIME], va.fare)], verbose=False)
rs_metrics["mae_without_time_features"] = float(mean_absolute_error(te.fare, nt.predict(te[NO_TIME])))
print("MAE with time features $%.2f vs without $%.2f -> time of day cuts error by %.1f%%"
      % (rs_metrics["mae_p50"], rs_metrics["mae_without_time_features"], 100 * (1 - rs_metrics["mae_p50"] / rs_metrics["mae_without_time_features"])))

te_eval = te.assign(p50=P[0.5], err=np.abs(P[0.5] - te.fare.values))
by_period = te_eval.groupby("period_name").agg(trips=("fare", "size"), actual=("fare", "median"), predicted=("p50", "median"), mae=("err", "mean")).reindex(PERIOD_NAMES)
print(by_period.round(2).to_string())
rs_by_period = {k: {c: float(v) for c, v in row.items()} for k, row in by_period.iterrows()}

train 5009476 | valid 263657 | test 665722 | test starts 2026-08-25 00:00:00
rate card: $8.61 + $3.76/mi


trained in 13.9s on cuda


rideshare {"n_test": 665722, "mae_p50": 6.4047, "r2_p50": 0.7827, "mape_p50": 0.242, "coverage_p10_p90": 0.7904, "median_band_width": 15.9591, "mae_baseline": 7.4099, "r2_baseline": 0.7572}


MAE with time features $6.40 vs without $6.63 -> time of day cuts error by 3.4%
                trips     actual  predicted   mae
period_name                                      
late night      72266  19.490000  19.709999  5.66
early morning   32437  20.459999  20.870001  5.81
morning rush    91406  18.070000  18.400000  5.79
midday         187367  19.330000  18.700001  6.75
evening rush   112130  19.389999  18.410000  7.08
evening        105617  19.080000  18.709999  6.35
night           64499  19.469999  19.580000  6.33


In [6]:
probe = pd.DataFrame([{"miles": 8, "hour": h, "dow": dw, "hour_of_week": dw * 24 + h, "period": PERIOD_OF_HOUR[h], "weekend": float(dw >= 5),
                       "month": 8, "airport": 0, "shared": 0, "lyft": 0} for dw in range(7) for h in range(24)], dtype="float32")
probe["p50"] = rs[0.5].predict(probe[RS_FEATURES])
print("median fare, 8-mile Uber ride, by request hour:")
print(probe.pivot(index="hour", columns="dow", values="p50").round(2).rename(columns=dict(enumerate("Mon Tue Wed Thu Fri Sat Sun".split()))).to_string())
wk = probe[probe.dow == 2].set_index("hour").p50
rs_metrics["weekday_hourly_swing_pct"] = float(100 * (wk.max() / wk.min() - 1))
print("\nWednesday: cheapest %d:00 ($%.2f), priciest %d:00 ($%.2f), swing %.0f%%" % (wk.idxmin(), wk.min(), wk.idxmax(), wk.max(), rs_metrics["weekday_hourly_swing_pct"]))
imp = pd.Series(rs[0.5].get_booster().get_score(importance_type="gain")).sort_values(ascending=False)
print("\nfeature importance (gain):"); print(imp.round(1).to_string())

median fare, 8-mile Uber ride, by request hour:
dow         Mon        Tue        Wed        Thu        Fri        Sat        Sun
hour                                                                             
0.0   28.020000  29.100000  29.940001  30.160000  30.740000  31.920000  32.250000
1.0   28.010000  28.809999  29.340000  29.549999  30.040001  30.990000  31.320000
2.0   28.360001  28.500000  28.990000  28.990000  29.290001  30.740000  31.080000
3.0   28.490000  28.219999  28.809999  28.660000  29.120001  30.850000  31.240000
4.0   29.330000  28.410000  28.660000  28.430000  28.850000  30.459999  31.030001
5.0   29.879999  29.370001  29.260000  29.180000  29.160000  28.639999  28.650000
6.0   31.740000  32.139999  31.370001  31.410000  30.370001  28.110001  28.070000
7.0   34.770000  35.529999  34.270000  34.580002  32.740002  29.420000  27.969999
8.0   35.240002  36.500000  35.459999  35.389999  33.450001  29.920000  28.170000
9.0   34.830002  36.299999  36.090000  35.630001  

## 2 · Airfares (U.S. DOT Consumer Airfare Report, Table 1a)

Quarterly average fares for every U.S. airport-pair market. Airport-level features (traffic and fare per mile) are computed **on training years only** and exported with the model so the app can price any pair of airports, including pairs the table never saw.

In [7]:
AF_URL = "https://data.transportation.gov/resource/tfrh-tu9e.csv?$limit=2000000"
af_path = download(AF_URL, f"{DATA}/airfare_table1a.csv")
af = pd.read_csv(af_path, low_memory=False)
print(af.shape); print(list(af.columns))

(262087, 31)
['tbl', 'year', 'quarter', 'citymarketid_1', 'citymarketid_2', 'city1', 'city2', 'airportid_1', 'airportid_2', 'airport_1', 'airport_2', 'nsmiles', 'passengers', 'fare', 'carrier_lg', 'large_ms', 'fare_lg', 'carrier_low', 'lf_ms', 'fare_low', 'location_1', 'location_1_address', 'location_1_city', 'location_1_state', 'location_1_zip', 'location_2', 'location_2_address', 'location_2_city', 'location_2_state', 'location_2_zip', 'tbl1apk']


In [8]:
af = af.rename(columns=str.lower)
need = ["year", "quarter", "airport_1", "airport_2", "city1", "city2", "nsmiles", "passengers", "fare"]
af = af[need].dropna()
af = af[(af.year >= 2015) & af.fare.between(20, 2000) & (af.nsmiles > 50)].copy()
af["t"] = af.year + (af.quarter - 1) / 4
test_from = af.t.max() - 0.75                                   # temporal split: the last 4 quarters are the test set
af_tr, af_te = af[af.t < test_from], af[af.t >= test_from]

# Airport coordinates: OurAirports (public domain), matched on IATA code.
oa = pd.read_csv(download("https://davidmegginson.github.io/ourairports-data/airports.csv", f"{DATA}/ourairports.csv"),
                 usecols=["iata_code", "latitude_deg", "longitude_deg", "type"], low_memory=False)
oa = oa.dropna(subset=["iata_code"]).sort_values("type").drop_duplicates("iata_code").set_index("iata_code")

# airport stats from training years only (no leakage), both directions of every pair
both = pd.concat([af_tr.rename(columns={"airport_1": "ap", "city1": "city"}), af_tr.rename(columns={"airport_2": "ap", "city2": "city"})])
st = both.groupby("ap").agg(pax=("passengers", "sum"), fare=("fare", "sum"), miles=("nsmiles", "sum"), city=("city", "first"))
st["fpm"] = st.fare / st.miles
st["log_pax"] = np.log1p(st.pax)
st = st.join(oa[["latitude_deg", "longitude_deg"]].rename(columns={"latitude_deg": "lat", "longitude_deg": "lon"}))
med = {"log_pax": float(st.log_pax.median()), "fpm": float(st.fpm.median())}
print("airports:", len(st), "| with coordinates:", int(st.lat.notna().sum()))

def af_features(df):
    o = st.reindex(df.airport_1); dd = st.reindex(df.airport_2)
    return pd.DataFrame({
        "nsmiles": df.nsmiles.values, "quarter": df.quarter.values, "year": df.year.values,
        "o_log_pax": o.log_pax.fillna(med["log_pax"]).values, "d_log_pax": dd.log_pax.fillna(med["log_pax"]).values,
        "o_fpm": o.fpm.fillna(med["fpm"]).values, "d_fpm": dd.fpm.fillna(med["fpm"]).values,
    }, dtype="float32")

AF_FEATURES = ["nsmiles", "quarter", "year", "o_log_pax", "d_log_pax", "o_fpm", "d_fpm"]
Xtr, Xte = af_features(af_tr), af_features(af_te)
va_mask = np.random.default_rng(SEED).random(len(Xtr)) < 0.1
afm = train_quantiles(Xtr[~va_mask], af_tr.fare.values[~va_mask], Xtr[va_mask], af_tr.fare.values[va_mask],
                      dict(n_estimators=300, max_depth=6, learning_rate=0.06, subsample=0.8))
af_base = LinearRegression().fit(af_tr[["nsmiles"]], af_tr.fare)
AP = {q: afm[q].predict(Xte) for q in QUANTILES}
print("train", len(af_tr), "| test", len(af_te), "| test from", round(test_from, 2))
af_metrics = interval_report("airfare", af_te.fare.values, AP[0.1], AP[0.5], AP[0.9], af_base.predict(af_te[["nsmiles"]]))

airports: 180 | with coordinates: 179


train 81583 | test 8281 | test from 2025.25
airfare {"n_test": 8281, "mae_p50": 32.6379, "r2_p50": 0.5129, "mape_p50": 0.127, "coverage_p10_p90": 0.7307, "median_band_width": 80.2686, "mae_baseline": 51.473, "r2_baseline": 0.121}


## 3 · Flight time-of-day factor (Expedia domestic fares, New Zealand)

The DOT airfare table has no departure times, so it can't say whether a 6 am flight costs less than a 5 pm one. This dataset can: about 162,000 Expedia fares with the departure time of every flight (Air New Zealand and Jetstar, 52 routes, 90 travel dates). To carry the *pattern* to U.S. routes without carrying New Zealand price levels or currency, the target is each fare divided by the median fare on the same route and travel date. The app multiplies the U.S. route-level estimate by this factor. The data has no booking date, so booking lead time is not modeled.

In [9]:
NZ_URL = "https://data.mendeley.com/public-files/datasets/w8wk4h9hxc/files/5d6d1fa8-7305-4cf8-b6ff-1196d1315a8c/file_downloaded"
nz = pd.read_csv(download(NZ_URL, f"{DATA}/nz_airfares.csv"), encoding="utf-8-sig")
nz.columns = ["date", "dep", "dep_time", "arr", "arr_time", "duration", "direct", "transit", "baggage", "airline", "fare"]
nz = nz.dropna(subset=["date", "dep", "arr", "dep_time", "fare"]).copy()
nz["date"] = pd.to_datetime(nz.date, format="%d/%m/%Y")
t = pd.to_datetime(nz.dep_time, format="%I:%M %p")
nz["dep_hour"] = (t.dt.hour + t.dt.minute / 60).astype("float32")
def minutes_of(s):
    h = pd.Series(s).str.extract(r"(\d+)h").fillna(0).astype(float)[0]
    m = pd.Series(s).str.extract(r"(\d+)m").fillna(0).astype(float)[0]
    return (h * 60 + m).values
nz["minutes"] = minutes_of(nz.duration)
nz["stops"] = nz.direct.map({"(Direct)": 0, "(1 stop)": 1, "(2 stops)": 2, "(3 stops)": 3}).astype("float32")
nz["route"] = nz.dep + "-" + nz.arr
nz = nz.dropna(subset=["stops"])
nz = nz[nz.fare.between(30, 3000) & (nz.minutes > 0)]
nz["rel"] = (nz.fare / nz.groupby(["route", "date"]).fare.transform("median")).astype("float32")
nz["dur_ratio"] = (nz.minutes / nz.groupby("route").minutes.transform("min")).astype("float32")
hr = nz.dep_hour.astype(int).values
nz["dow"] = nz.date.dt.dayofweek.astype("float32")
nz["period"] = PERIOD_OF_HOUR[hr]
nz["weekend"] = (nz.dow >= 5).astype("float32")
nz["period_name"] = nz.period.astype(int).map(dict(enumerate(PERIOD_NAMES)))
print(len(nz), "fares |", nz.route.nunique(), "routes |", nz.date.nunique(), "travel dates")
print("median fare vs same route & day, by departure period (nonstop only):")
print(nz[nz.stops == 0].groupby("period_name").rel.agg(["median", "size"]).reindex(PERIOD_NAMES).round(3).to_string())

162809 fares | 32 routes | 90 travel dates
median fare vs same route & day, by departure period (nonstop only):
               median    size
period_name                  
late night        NaN     NaN
early morning   0.421  1102.0
morning rush    0.520  4430.0
midday          0.535  7304.0
evening rush    0.546  4398.0
evening         0.543  2473.0
night             NaN     NaN


In [10]:
FT_FEATURES = ["dep_hour", "period", "dow", "weekend", "stops", "dur_ratio"]
last = nz.date.max() - pd.Timedelta(days=21)                 # temporal split: the last 3 weeks of travel dates
ftr, fte = nz[nz.date < last], nz[nz.date >= last]
fva_mask = np.random.default_rng(SEED).random(len(ftr)) < 0.1
ft = train_quantiles(ftr[FT_FEATURES][~fva_mask], ftr.rel[~fva_mask], ftr[FT_FEATURES][fva_mask], ftr.rel[fva_mask],
                     dict(n_estimators=200, max_depth=5, learning_rate=0.08, subsample=0.8))
FP = {q: ft[q].predict(fte[FT_FEATURES]) for q in QUANTILES}
ft_metrics = interval_report("flight time-of-day (relative price)", fte.rel.values, FP[0.1], FP[0.5], FP[0.9], np.ones(len(fte)))
route_day_median = fte.fare / fte.rel
ft_metrics["mae_fare_nzd"] = float(mean_absolute_error(fte.fare, route_day_median * FP[0.5])); ft_metrics["mae_fare_nzd_no_time"] = float(mean_absolute_error(fte.fare, route_day_median))
print("fare MAE with time-of-day factor NZ$%.2f vs flat route-day median NZ$%.2f" % (ft_metrics["mae_fare_nzd"], ft_metrics["mae_fare_nzd_no_time"]))
grid = pd.DataFrame([{"dep_hour": h, "period": PERIOD_OF_HOUR[h], "dow": 2, "weekend": 0, "stops": 0, "dur_ratio": 1} for h in range(5, 24)], dtype="float32")
grid["factor"] = ft[0.5].predict(grid[FT_FEATURES])
print("\nnonstop, Wednesday — price factor by departure hour:"); print(grid.set_index("dep_hour").factor.round(3).to_string())

flight time-of-day (relative price) {"n_test": 28963, "mae_p50": 0.1704, "r2_p50": 0.3047, "mape_p50": 0.1483, "coverage_p10_p90": 0.8032, "median_band_width": 0.5205, "mae_baseline": 0.2026, "r2_baseline": -0.0037}
fare MAE with time-of-day factor NZ$63.75 vs flat route-day median NZ$78.89

nonstop, Wednesday — price factor by departure hour:
dep_hour
5.0     0.604
6.0     0.374
7.0     0.500
8.0     0.531
9.0     0.610
10.0    0.575
11.0    0.586
12.0    0.544
13.0    0.506
14.0    0.490
15.0    0.497
16.0    0.475
17.0    0.563
18.0    0.539
19.0    0.601
20.0    0.507
21.0    0.420
22.0    0.384
23.0    0.384


## 4 · Export and parity check

Each quantile model is converted to compact JSON and re-scored with the pure-Python reference evaluator; the largest difference from XGBoost must be under one cent. Twenty "golden" rows per model are exported so the TypeScript tests can assert the same numbers.

In [11]:
def export(name, models, features, Xtest, metrics, data, extra=None):
    out = {"name": name, "version": time.strftime("%Y-%m-%d"), "features": features, "quantiles": {}, "metrics": metrics, "data": data,
           "xgboost": xgb.__version__, "trained_on": DEVICE}
    golden_X = np.asarray(Xtest[features].iloc[:20], dtype=np.float32)
    golden = {"X": golden_X.tolist()}
    worst = 0.0
    for q, m in models.items():
        cm = compact(m); key = f"p{int(q * 100)}"
        ref = m.predict(Xtest[features].iloc[:2000]); mine = predict_compact(cm, Xtest[features].iloc[:2000])
        worst = max(worst, float(np.max(np.abs(ref - mine))))
        out["quantiles"][key] = cm
        golden[key] = [round(float(v), 4) for v in m.predict(golden_X)]
    out["parity_max_abs_diff"] = worst; out["golden"] = golden
    if extra: out.update(extra)
    path = f"{OUT}/{name}.json"
    with open(path, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print(name, "parity max |diff| =", worst, "|", round(os.path.getsize(path) / 1024), "KB ->", path)
    assert worst < 0.01, "compact evaluator disagrees with XGBoost"

export("rideshare_fare", rs, RS_FEATURES, te, rs_metrics,
       {"source": "NYC TLC High Volume FHV trip records", "url": "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page",
        "months": MONTHS, "rows_train": int(len(tr)), "target": "base_passenger_fare (USD, before tax/tolls/fees/tip)",
        "region": "New York City", "rate_card": {"intercept": float(base.intercept_), "per_mile": float(base.coef_[0])}},
       {"periods": [{"name": n, "from_hour": a, "to_hour": b} for n, a, b in PERIODS], "by_period": rs_by_period})

airports = st.dropna(subset=["lat", "lon"]).reset_index()
export("airfare", afm, AF_FEATURES, Xte, af_metrics,
       {"source": "U.S. DOT Consumer Airfare Report Table 1a", "url": "https://data.transportation.gov/d/tfrh-tu9e",
        "years": [int(af.year.min()), int(af.year.max())], "test_from": float(test_from), "rows_train": int(len(af_tr)),
        "target": "average fare for the airport-pair market and quarter (USD)"},
       {"airports": [{"code": r.ap, "city": r.city, "lat": round(r.lat, 4), "lon": round(r.lon, 4),
                      "log_pax": round(float(r.log_pax), 4), "fpm": round(float(r.fpm), 5)} for r in airports.itertuples()],
        "airport_defaults": med, "latest_year": int(af.year.max())})
export("flight_time_of_day", ft, FT_FEATURES, fte, ft_metrics,
       {"source": "Airfares in New Zealand (Expedia, Sep-Dec 2019), Mendeley Data, CC BY 4.0", "url": "https://data.mendeley.com/datasets/w8wk4h9hxc/1",
        "rows_train": int(len(ftr)), "target": "fare / median fare for the same route and travel date"},
       {"periods": [{"name": n, "from_hour": a, "to_hour": b} for n, a, b in PERIODS]})
print(sorted(os.listdir(OUT)))

rideshare_fare parity max |diff| = 0.0001617400216673559 | 3422 KB -> /net/home/chatran/smile-price-models/models/rideshare_fare.json


airfare parity max |diff| = 0.00020262200837350974 | 1936 KB -> /net/home/chatran/smile-price-models/models/airfare.json


flight_time_of_day parity max |diff| = 7.76905229482594e-07 | 734 KB -> /net/home/chatran/smile-price-models/models/flight_time_of_day.json
['airfare.json', 'flight_time_of_day.json', 'rideshare_fare.json']
